[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/04_handwritten_digits_binary.ipynb)

# Bài 04 — Bài toán phân loại 2 chữ số viết tay

Bài mẫu phân loại **nhị phân** hai chữ số trong bộ dữ liệu `digits` của scikit-learn, dùng `LogisticRegression` để tìm nghiệm tối ưu. Phần bổ sung: **nhìn vào bên trong model** — trọng số của Logistic Regression trên ảnh chính là một "ảnh mẫu" có thể diễn giải được.

<img src="images/digits-dataset.png" style="width:45%;height:45%;">

## Load dữ liệu từ sklearn

Bộ dữ liệu có sẵn trong thư viện, không cần tải về: 1 797 ảnh 8×8 của chữ số 0–9 (bộ MNIST 28×28 lớn hơn nhiều — chương 29).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

from ml_course.visualization import plot_confusion_matrix, set_style

set_style()
digits = load_digits()
print("Image Data Shape:", digits.data.shape)     # mỗi ảnh 8×8 được duỗi thành vector 64 chiều
print("Label Data Shape:", digits.target.shape)

## Trực quan hoá dữ liệu

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(16, 2.2))
for ax, image, label in zip(axes, digits.images[:10], digits.target[:10]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(f"Training: {label}")
    ax.axis("off")
plt.show()

Có thể chọn tuỳ ý 2 chữ số. Bài này dùng **8 và 9** vì hai số này dễ bị nhầm lẫn với nhau. Mã hoá nhãn thành 0/1: số 8 → 0, số 9 → 1.

In [ ]:
mask = np.isin(digits.target, [8, 9])
X = digits.data[mask] / 16.0                       # đưa pixel về [0, 1]
y = (digits.target[mask] == 9).astype(int)
print(X.shape, "| số 8:", np.sum(y == 0), "| số 9:", np.sum(y == 1))

## Chia dữ liệu thành tập Train và tập Test

Tỉ lệ 75% train / 25% test, **stratify** để giữ tỉ lệ 2 lớp:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

## 4 bước sử dụng scikit-learn để phân loại

**Bước 1:** import model; **Bước 2:** tạo đối tượng model; **Bước 3:** train với `fit(X, y)`; **Bước 4:** dự đoán với `predict`.

In [ ]:
from sklearn.linear_model import LogisticRegression          # Bước 1

logistic_regr = LogisticRegression(max_iter=1000)             # Bước 2
logistic_regr.fit(X_train, y_train)                           # Bước 3

print("Dự đoán 1 ảnh  :", logistic_regr.predict(X_test[0].reshape(1, -1)))   # Bước 4
print("Dự đoán 10 ảnh :", logistic_regr.predict(X_test[:10]))
predictions = logistic_regr.predict(X_test)

*(bổ sung)* Logistic Regression cho **xác suất**, không chỉ nhãn — hữu ích để biết model "chắc chắn" đến đâu:

In [ ]:
proba = logistic_regr.predict_proba(X_test[:5])
for p, t in zip(proba, y_test[:5]):
    print(f"P(số 8) = {p[0]:.3f} | P(số 9) = {p[1]:.3f} | nhãn thật: {'9' if t else '8'}")

## Đo hiệu suất mô hình

In [ ]:
score = logistic_regr.score(X_test, y_test)
print(f"Accuracy: {score:.4f}")

## Confusion Matrix

Confusion matrix đánh giá kết quả phân loại theo từng lớp: hàng là nhãn thật, cột là nhãn dự đoán; đường chéo là dự đoán đúng. Chi tiết về các metric (precision, recall, F1...) ở chương 14.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)
print(cm)
fig, ax = plt.subplots(figsize=(3.5, 3))
plot_confusion_matrix(cm, labels=["8", "9"], ax=ax, title=f"Accuracy: {score:.3f}")
plt.show()

> 📝 *So với bản gốc:* dùng hàm vẽ `plot_confusion_matrix` của khoá học (không cần seaborn) và không ghi file ảnh ra thư mục làm việc.

## Nhìn vào bên trong model *(bổ sung)*

Model tính $z = \mathbf w^\top\mathbf x + b$ với $\mathbf x$ là 64 pixel → mỗi pixel có một trọng số. Vẽ $\mathbf w$ thành ảnh 8×8: pixel **xanh** (trọng số dương) đẩy về "số 9", pixel **đỏ** (âm) đẩy về "số 8".

In [ ]:
w = logistic_regr.coef_.reshape(8, 8)
mean8 = X_train[y_train == 0].mean(0).reshape(8, 8)
mean9 = X_train[y_train == 1].mean(0).reshape(8, 8)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
axes[0].imshow(mean8, cmap="gray_r")
axes[0].set_title("Ảnh trung bình số 8")
axes[1].imshow(mean9, cmap="gray_r")
axes[1].set_title("Ảnh trung bình số 9")
lim = np.abs(w).max()
im = axes[2].imshow(w, cmap="RdBu", vmin=-lim, vmax=lim)
axes[2].set_title("Trọng số w (xanh → 9, đỏ → 8)")
fig.colorbar(im, ax=axes[2])
for ax in axes:
    ax.axis("off")
plt.show()

Trọng số âm tập trung ở **nửa dưới bên trái** — nơi số 8 có nét cong khép kín còn số 9 thì không. Model đã học đúng đặc điểm phân biệt! Đây là ví dụ về **tính diễn giải được** của model tuyến tính (chương 36).

## Xem các ảnh bị dự đoán sai *(bổ sung)*

In [ ]:
wrong = np.flatnonzero(predictions != y_test)
proba_all = logistic_regr.predict_proba(X_test)[:, 1]
fig, axes = plt.subplots(1, max(len(wrong), 1), figsize=(2.2 * max(len(wrong), 1), 2.4), squeeze=False)
for ax, i in zip(axes[0], wrong):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"thật {'9' if y_test[i] else '8'}\nP(9) = {proba_all[i]:.2f}", fontsize=9)
    ax.axis("off")
plt.show()
print(f"{len(wrong)} ảnh bị sai trên {len(y_test)}")

Các ảnh sai thường viết "lạ" thật sự — đôi khi cả người cũng phân vân (error analysis — chương 05, bài 05).

## 🧠 Kiểm tra nhanh

1. Vì sao chia pixel cho 16?
2. Trọng số của một pixel là số dương lớn — pixel đó sáng (có nét) thì model nghiêng về lớp nào?
3. Thay 8/9 bằng 0/1 — accuracy sẽ cao hơn hay thấp hơn? Vì sao?

## Tài liệu tham khảo

[1] *Logistic Regression using Python (scikit-learn)* — Michael Galarnyk · [2] scikit-learn — [Recognizing hand-written digits](https://scikit-learn.org/stable/auto_examples/classification/plot_digits_classification.html)

---
<sub>Bài gốc: *Bài toán phân loại 2 chữ số viết tay* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License). Đã bổ sung: chuẩn hoá pixel, stratify, xác suất dự đoán, trực quan hoá trọng số, xem ảnh dự đoán sai.</sub>